In [1]:
# %pip install kagglehub

In [2]:
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter

## Modification of existing dataset Airlines: 
    - source: https://www.kaggle.com/datasets/wenxingdi/data-expo-2009-airline-on-time-data?resource=download
    - added new columns: previous_departure_delay indicating how much delayed was the plane's previous flight, 
      scheduled_turnaround indicating how much scheduled time between flights the plane has

    - droped features (below - columns_to_drop) not relevant to the prediction such as TaxiIn 
    - dropped features related to y_pred such as WeatherDelay (normally at the time of prediction such features would not be present)
    - filtered only flights that were not cancelled or diverted 
Features present in the modified dataset: ['Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime',
       'CRSArrTime', 'CRSElapsedTime', 'ArrDelay', 'Distance',
      'previous_dep_delay', 'scheduled_turnaround', 'origin_state',
      'origin_lat', 'origin_long', 'dest_state', 'dest_lat', 'dest_long'] 
where ArrDelay is the targed variable

In [4]:
columns_to_drop = [
        'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay','LateAircraftDelay',
        'TaxiIn', 'TaxiOut', 'DepDelay',  
        'AirTime', 'ActualElapsedTime', 'UniqueCarrier', 
        'ArrTime','Year',
        'CancellationCode','Diverted','Cancelled',
        'FlightNum','TailNum',
        'Origin','Dest'
    ]

In [5]:
pd.set_option('display.max_columns', None)

In [6]:
path = "airports.csv" 

df_airports = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "wenxingdi/data-expo-2009-airline-on-time-data",
  path
)
path2 = "2008.csv" 
df0 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "wenxingdi/data-expo-2009-airline-on-time-data",
  path2
)

path3="carriers.csv"
df_carriers = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "wenxingdi/data-expo-2009-airline-on-time-data",
  path3
)


In [7]:
def previous_flight_features(df):
    df = df.copy()
    
    def hours_transform(time):
        if pd.isna(time):
            return np.nan
        time = int(time)
        hours = time // 100
        mins = time % 100
        t=hours * 60 + mins
        return t

    df['DepTime_mins'] = df['DepTime'].apply(hours_transform)
    df['CRSDepTime_mins'] = df['CRSDepTime'].apply(hours_transform)
    df['CRSArrTime_mins'] = df['CRSArrTime'].apply(hours_transform)
    

    df = df.sort_values(by=['Year', 'Month', 'DayofMonth', 'TailNum', 'DepTime_mins'])
    
    grouping = ['Year', 'Month', 'DayofMonth', 'TailNum']
    
    df['previous_dep_delay'] = df.groupby(grouping)['DepDelay'].shift(1)
    df['previous_dep_delay'] = df['previous_dep_delay'].fillna(0)
    df['prev_CRSArrTime'] = df.groupby(grouping)['CRSArrTime_mins'].shift(1)
    
    df['scheduled_turnaround'] = df['CRSDepTime_mins'] - df['prev_CRSArrTime']
    
    df['scheduled_turnaround'] = df['scheduled_turnaround'].fillna(60*24)
    df.loc[df['scheduled_turnaround'] < 0, 'scheduled_turnaround'] += 24 * 60
    df = df.drop(columns=['DepTime_mins', 'CRSDepTime_mins', 'CRSArrTime_mins', 'prev_CRSArrTime'])

    df.loc[df['previous_dep_delay'] < 0, 'previous_dep_delay'] = 0
    df.loc[df['DepDelay'] < 0, 'DepDelay'] = 0
    
    return df

df = previous_flight_features(df0)
df

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,previous_dep_delay,scheduled_turnaround
465489,2008,1,1,2,714.0,715,925.0,923,9E,5615,80019E,131.0,128.0,92.0,2.0,0.0,XNA,MSP,596,12.0,27.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,1440.0
463104,2008,1,1,2,1657.0,1605,1804.0,1650,9E,4722,80019E,67.0,45.0,23.0,74.0,52.0,MSP,RST,76,3.0,41.0,0,NaN,0,0.0,0.0,22.0,0.0,52.0,0.0,402.0
463589,2008,1,1,2,1829.0,1720,1908.0,1812,9E,4743,80019E,39.0,52.0,20.0,56.0,69.0,RST,MSP,76,11.0,8.0,0,NaN,0,0.0,0.0,0.0,0.0,56.0,52.0,30.0
465147,2008,1,1,2,1935.0,1915,2303.0,2252,9E,5601,80019E,148.0,157.0,120.0,11.0,20.0,MSP,GSO,924,6.0,22.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN,69.0,63.0
465184,2008,1,1,2,622.0,620,741.0,728,9E,5603,80059E,79.0,68.0,42.0,13.0,2.0,DSM,MSP,232,15.0,22.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,1440.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2291645,2008,4,30,3,NaN,1255,NaN,1500,AA,1423,NaN,NaN,245.0,NaN,NaN,NaN,STL,LAX,1593,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,0.0,1440.0
2297180,2008,4,30,3,NaN,620,NaN,1210,AA,1646,NaN,NaN,230.0,NaN,NaN,NaN,SFO,STL,1736,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,0.0,1440.0
2300746,2008,4,30,3,NaN,1350,NaN,1845,AA,1774,NaN,NaN,175.0,NaN,NaN,NaN,SNA,DFW,1205,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,0.0,1440.0
2343984,2008,4,30,3,NaN,1850,NaN,2323,CO,732,NaN,NaN,213.0,NaN,NaN,NaN,IAH,LGA,1416,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN,0.0,1440.0


In [8]:
def airline_stream(df):
    
    columns_to_drop = [
        'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay','LateAircraftDelay',
        'TaxiIn', 'TaxiOut', 'ArrDelay', 'DepDelay',  
        'AirTime', 'ActualElapsedTime', 'UniqueCarrier', 
        'ArrTime','Year',
        'CancellationCode','Diverted','Cancelled',
        'FlightNum','TailNum',
        'Origin','Dest'
        
        
    ]
    
    for row in df:
        x = dict(row)
        is_cancelled = int(x.get('Cancelled', 0)) == 1
        is_diverted = int(x.get('Diverted', 0)) == 1
        if is_cancelled or is_diverted:
            continue
            
        y = x.get('ArrDelay')
        for col in columns_to_drop:
            x.pop(col, None)
        
        yield x, y

In [9]:
df_m = df.merge(
    df_airports[['iata', 'state','lat','long']], 
    left_on='Origin', 
    right_on='iata', 
    how='inner'
)
df_m= df_m.rename(columns={'state': 'origin_state','lat':'origin_lat','long':'origin_long'}).drop(columns='iata')


In [10]:
df_merged = df_m.merge(
    df_airports[['iata', 'state','lat','long']], 
    left_on='Dest', 
    right_on='iata', 
    how='inner'
)
df_merged = df_merged.rename(columns={'state': 'dest_state','lat':'dest_lat','long':'dest_long'}).drop(columns='iata')

In [11]:
df_merged=df_merged.sort_values(by=['Year', 'Month', 'DayofMonth', 'CRSDepTime'])

In [12]:
df_merged["Diverted"].unique()

array([0, 1])

In [13]:
df_final=df_merged[df_merged["Cancelled"]==0]
df_final=df_final[df_final["Diverted"]==0]

In [14]:
df_final["Diverted"].unique()

array([0])

In [15]:
df_final = df_final.drop(columns=columns_to_drop)

In [16]:
df_final.head(10)

,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,CRSArrTime,CRSElapsedTime,ArrDelay,Distance,previous_dep_delay,scheduled_turnaround,origin_state,origin_lat,origin_long,dest_state,dest_lat,dest_long
9043,1,1,2,2400.0,10,737,267.0,-3.0,1979,82.0,248.0,CA,33.942536,-118.408074,MI,42.212059,-83.348836
10680,1,1,2,5.0,15,823,308.0,-19.0,2521,7.0,980.0,CA,38.695422,-121.590767,NY,40.639751,-73.778926
5658,1,1,2,19.0,25,709,284.0,-24.0,2153,0.0,1440.0,AZ,33.434167,-112.008056,NY,40.639751,-73.778926
8347,1,1,2,45.0,25,535,190.0,12.0,1431,0.0,1440.0,CA,38.695422,-121.590767,TX,32.895951,-97.037200
4481,1,1,2,26.0,30,444,194.0,24.0,1449,0.0,1440.0,AK,61.174320,-149.996186,WA,47.448982,-122.309313
10085,1,1,2,29.0,30,825,295.0,-27.0,2248,0.0,1440.0,NV,36.080361,-115.152333,NY,40.639751,-73.778926
10167,1,1,2,33.0,30,605,215.0,11.0,1536,0.0,1440.0,CA,33.942536,-118.408074,MN,44.880547,-93.216922
11819,1,1,2,21.0,30,831,301.0,-23.0,2430,0.0,1440.0,CA,34.056000,-117.601194,NY,40.639751,-73.778926
5603,1,1,2,33.0,35,603,208.0,-12.0,1635,0.0,1440.0,CA,37.619002,-122.374843,TX,29.980472,-95.339722
11072,1,1,2,30.0,35,420,165.0,-15.0,1189,0.0,1440.0,FL,28.428889,-81.316028,PR,18.439417,-66.001833


In [17]:
df_final.columns


Index(['Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime',
       'CRSArrTime', 'CRSElapsedTime', 'ArrDelay', 'Distance',
       'previous_dep_delay', 'scheduled_turnaround', 'origin_state',
       'origin_lat', 'origin_long', 'dest_state', 'dest_lat', 'dest_long'],
      dtype='object')

In [18]:
df_final.to_csv("Airplanes_modified.csv", header=True,index=False)

# Taxi

In [6]:
df_taxi=pd.read_csv("taxi_dataset.csv")
df_taxi=df_taxi.sort_values("pickup_datetime")
df_taxi.head(10)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
96469,id0190469,2,2016-01-01 00:00:17,2016-01-01 00:14:26,5,-73.981743,40.719158,-73.938828,40.829182,N,849
223872,id1665586,1,2016-01-01 00:00:53,2016-01-01 00:22:27,1,-73.985085,40.747166,-73.958038,40.717491,N,1294
713067,id1210365,2,2016-01-01 00:01:01,2016-01-01 00:07:49,5,-73.965279,40.801041,-73.947479,40.815170,N,408
652463,id3888279,1,2016-01-01 00:01:14,2016-01-01 00:05:54,1,-73.982292,40.751331,-73.991341,40.750340,N,280
722901,id0924227,1,2016-01-01 00:01:20,2016-01-01 00:13:36,1,-73.970108,40.759800,-73.989357,40.742989,N,736
1116520,id2294362,2,2016-01-01 00:01:33,2016-01-01 00:13:25,1,-73.984993,40.773891,-73.936493,40.847771,N,712
404427,id1078247,2,2016-01-01 00:01:37,2016-01-01 00:03:31,1,-73.973335,40.764072,-73.974854,40.761734,N,114
475569,id3609443,1,2016-01-01 00:01:47,2016-01-01 00:21:51,2,-73.993103,40.752632,-73.953903,40.816540,N,1204
1019552,id2914314,2,2016-01-01 00:02:06,2016-01-01 00:23:56,1,-73.985443,40.735710,-73.957489,40.811611,N,1310
300611,id3675972,2,2016-01-01 00:02:45,2016-01-01 00:10:10,1,-73.993736,40.741760,-74.004669,40.745468,N,445


In [7]:
df_taxi.to_csv("taxi_dataset_ordered.csv", header=True,index=False)